### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re part of the Psittaciformes order, right? There are different types like macaws, cockatoos, parakeets, etc. I know some can mimic human speech, but not all of them do. So why do they do that?\n\nFirst, maybe it\'s related to their social behavior. Parrots are social birds, often living in flocks in the wild. Communication within the flock is important for things like finding food, warning of predators, or coordinating movements. So maybe talking is a form of communication. But in captivity, when they interact with humans, they might learn to mimic human speech as a way to communicate with their human companions.\n\nAnother angle is their vocal anatomy. Parrots have a syrinx, which is the vocal organ in birds, but maybe their syrinx is more developed or structured in a way that allows for a wider range of sounds. Also, their tongue structure might play

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user mentioned Boston, I need to call this function with "Boston" as the location. I should make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'p2d925f95', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 154, 'total_tokens': 250, 'completion_time': 0.142964177, 'completion_tokens_details': {'reasoning_tokens': 72}, 'prompt_time': 0.006013977, 'prompt_tokens_details': None, 'queue_time': 0.052632507, 'total_time': 0.148978154}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': N

### Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call that function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': 'da7bkd7jx', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 153, 'total_tokens': 257, 'completion_time': 0.164752183, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.006190757, 'prompt_tokens_details': None, 'queue_time': 0.163693021, 'total_time': 0.17094294}, 'model_name': 'qwen/qwen3-32b', 'syste